# 15 · Bonus Capstone — Stock the Shelves
**BONUS** · ⏱ 90 min · 🧪 3 exercises · ⭐ 850 XP

*A real-world RL project end to end — retail inventory replenishment, from sales data and a classic planner to an agent judged on a year it has never seen.*

> **First time?** Open **00 · START HERE** in the same folder. Then run the cell below to connect this notebook to your Drive folder — your progress and XP are saved there.

In [ ]:
#@title ▶ **Run me first** — connect this notebook to your Reward Signal folder
#@markdown Mounts Google Drive and finds the `RewardSignal` folder (progress, XP and your files live there).
#@markdown Leave `ROOT` empty unless you put the folder somewhere unusual, e.g. `/content/drive/MyDrive/courses/RewardSignal`.
ROOT = ""  #@param {type:"string"}
LESSON = "15-bonus-inventory"
import os, sys

def _rs_locate(root=""):
    tries = [root, os.environ.get("RS_ROOT", "")]
    try:
        from google.colab import drive  # only exists in Colab
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        base = "/content/drive/MyDrive"
        tries += [f"{base}/RewardSignal", f"{base}/Colab Notebooks/RewardSignal", f"{base}/RewardSignal/RewardSignal"]
        tries += [os.path.join(base, d, "RewardSignal") for d in sorted(os.listdir(base)) if not d.startswith(".")]
        tries += ["/content/RewardSignal"]
    except ImportError:  # local Jupyter: the folder this notebook sits in
        tries += [os.getcwd(), os.path.dirname(os.getcwd())]
    for t in tries:
        if t and os.path.isfile(os.path.join(t, "runtime", "rlcolab.py")):
            return os.path.abspath(t)
    raise FileNotFoundError("Couldn't find the RewardSignal folder. Upload it to My Drive "
                            "(see its README.md) or type its full path into ROOT, then run this cell again.")

RS_ROOT = _rs_locate(ROOT.strip())
sys.dont_write_bytecode = True  # no __pycache__ folders in your Drive
if os.path.join(RS_ROOT, "runtime") not in sys.path:
    sys.path.insert(0, os.path.join(RS_ROOT, "runtime"))
import rlcolab
rlcolab.setup(RS_ROOT, lesson=LESSON)

Every evening a store manager answers the same question for thousands of products:
**how many do I order from the supplier?** Order too little and customers leave empty-handed.
Order too much and money sits on the shelf, then goes stale. The supplier needs two days, the
weekend is busy, marketing has a promotion next Thursday, and nobody knows tomorrow's demand.

This bonus capstone is a complete, realistic RL project on that problem. No game, no grid:
a simulator built from **sales data**, a **classic operations-research baseline** that you must
beat, an agent, and an honest **out-of-time evaluation** on a year of data it never trained on.
You can run it on the bundled two-year sales file — or on your own export.

## 01 · Why this is an RL problem

Replenishment has every ingredient from lesson 01:

- **Sequential decisions** — today's order changes tomorrow's stock, which changes what you
  should order tomorrow.
- **Delayed consequences** — with a two-day lead time, today's order cannot help today's
  customers. The reward for a good decision arrives days later, mixed with the effects of
  every other order.
- **Uncertainty** — demand is noisy, seasonal and promotional. You choose under risk, and the
  outcome of one day says little about whether the decision was good.
- **A clear scalar goal** — profit. Rarely does the reward hypothesis fit this naturally.

The simulator lives in the bundled `inventory` module. One `step` is one day:

1. an order of `env.order_sizes[a]` units is placed (a fixed fee if it is non-zero)
2. deliveries ordered `lead_time` days ago arrive; anything above the shelf capacity is refused
3. customers arrive: sales = min(stock, demand); the rest is **lost**
4. a few percent of the remaining stock **spoils** overnight
5. holding cost is charged on what is left

In [ ]:
# 📎 Example from the briefing — run it, change it, break it.
from inventory import InventoryEnv
env = InventoryEnv()                      # one simulated year of a fresh bakery item
obs, info = env.reset(seed=0)             # obs: int state in [0, env.n_states)
obs, reward, terminated, truncated, info = env.step(4)    # order env.order_sizes[4] units
print(info["sales"], info["lost_sales"], info["spoiled"], info["costs"])

## 02 · The MDP

| | In the simulator | Why |
|---|---|---|
| **State** | inventory *position* (on hand **+ in transit**), weekday, *promo soon* flag | what you own, and what demand is coming |
| **Action** | order 0, 6, 12, …, 48 units | a small discrete set, like real case packs |
| **Reward** | the day's profit | revenue − cost of goods − waste − fees − holding − stockout penalty |
| **γ** | 0.9 – 0.97 | an order's effects fade within a week or two |

`obs="tabular"` packs the state into one integer (`env.decode_state(s)` unpacks it).
`obs="vector"` gives 14 features (stock levels, weekday one-hot, promo flags, recent demand
level) for function approximation.

In [ ]:
#@title 🧠 **Knowledge check 1** · +25 XP — run, then pick an answer
rlcolab.quiz("15-bonus-inventory#0", {"q": "Why does the state use inventory position (on hand + in transit) rather than just the stock on the shelf?", "options": ["It is easier to compute", "Two shelves with equal stock but different deliveries on the way need different orders — without the pipeline the state is not Markov", "Because holding cost is charged on in-transit units"], "answer": 1, "explain": "With a lead time, units already ordered will arrive regardless of today's action. Ignoring them makes the agent order twice for the same gap. Operations research reached the same conclusion decades ago: base-stock policies act on inventory position."})

## 03 · The bar to beat: the planner

Long before RL, operations research solved simplified versions of this problem exactly.

The **newsvendor** sells a perishable product for one day. If an unsold unit costs *cₒ* (overage)
and a missing unit costs *cᵤ* (lost margin + goodwill), the optimal order is a quantile of the
demand distribution:

```text
q* = F⁻¹( cᵤ / (cᵤ + cₒ) )
```

With a margin of 4 + a goodwill penalty of 1 against an overage of 6, that ratio is 5/11 ≈ 0.45:
stock slightly *below* median demand.

For many periods with a lead time and a fixed fee per order, the classic answer is an
**(s, S) policy**: *whenever the inventory position drops to s or below, order up to S*. The gap
S − s batches orders so you do not pay the fee every day. Two numbers, tuned on history by grid
search, and it is remarkably hard to beat.

> **🔑 KEY IDEA** · **Beat a tuned baseline, or you have nothing**
>
> An RL agent that is "profitable" means nothing on its own. The question a business asks is
> *how much better than what we do today?* Always tune the classic policy **as well as you can**,
> on the same data, and report the agent relative to it.

---
### 🧪 Exercise 1 · Know your demand, then beat a planner · ⭐ **250 XP**

Before any RL, do what an operations analyst would do: **look at the data** and **tune the classic policy**. Its profit is the bar every agent must clear.

1. `weekday_profile(history)` — using **non-promo days only**, the mean demand of each weekday (Mon…Sun) divided by the mean over all those days. Returns 7 numbers; a value of 2.0 means that weekday sells twice an average day.
2. `promo_lift(history)` — how much a promotion multiplies demand, *corrected for the weekday*: sum of demand on promo days ÷ sum of the expected demand on those same days, where the expectation for a day is the mean non-promo demand of its weekday.
3. `grid_search_sS(env, s_grid, S_grid, seeds)` — try every pair with `S > s`, score each with `evaluate_policy(env, lambda e: s_S_policy(e, s, S), seeds=seeds)` and return `(best_s, best_S, best_profit, table)` with `table[(s, S)] = profit`.

`history` is a dict of numpy arrays `demand`, `dow` (0 = Monday) and `promo` (1 on promo days), as returned by `load_demand_csv` or `generate_sales_history`.

The demo explores the bundled two-year sales file, then tunes on **4 simulated training years** and reports profit on **20 held-out test years**.

**Pass:** both statistics match hand-computed cases and recover the simulator's patterns, the grid search is exact, and the tuned (s, S) earns **≥ 13,000** per held-out test year.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 15-bonus-inventory/01-baseline-planner
import numpy as np
import rlhud
from inventory import (InventoryEnv, load_demand_csv, sample_csv_path, s_S_policy,
                       evaluate_policy, run_policy, DOW_NAMES)


def weekday_profile(history):
    """Mean demand per weekday divided by the mean over all days, using NON-promo days only.

    history: dict with numpy arrays "demand", "dow" (0 = Mon) and "promo" (1 on promo days).
    Returns a length-7 array that averages ~1.0 (Saturday ≈ 2.0 means twice a normal day)."""
    d, dow, promo = (np.asarray(history[k]) for k in ("demand", "dow", "promo"))
    # TODO: keep only non-promo days, average demand per weekday, divide by their overall mean
    raise NotImplementedError


def promo_lift(history):
    """How much a promotion multiplies demand, corrected for the weekday mix.

    lift = (total demand on promo days) / (total demand you would EXPECT on those days
    without a promotion), where the expectation for a day is the mean non-promo demand
    of its weekday."""
    d, dow, promo = (np.asarray(history[k]) for k in ("demand", "dow", "promo"))
    # TODO: base[k] = mean non-promo demand on weekday k (a length-7 array)
    # TODO: lift = sum of demand on promo days / sum of base[dow] over those same days
    raise NotImplementedError


def grid_search_sS(env, s_grid, S_grid, seeds):
    """Try every (s, S) with S > s; score = mean episode profit over `seeds`
    (use inventory.evaluate_policy with inventory.s_S_policy).

    Returns (best_s, best_S, best_profit, table) where table[(s, S)] = mean profit."""
    table = {}
    # TODO: loop over s in s_grid and S in S_grid (skip S <= s)
    #   profit = evaluate_policy(env, lambda e: s_S_policy(e, s, S), seeds=seeds)
    #   remember it in table[(s, S)] and keep track of the best
    raise NotImplementedError


TRAIN_SEEDS = range(0, 4)        # demand years we tune on
TEST_SEEDS = range(100, 120)     # demand years we never looked at while tuning
S_GRID_LOW = range(20, 81, 10)   # reorder points s
S_GRID_HIGH = range(30, 121, 10)  # order-up-to levels S


if __name__ == "__main__":
    hist = load_demand_csv(sample_csv_path())
    n = len(hist["demand"])
    rlhud.plot(hist["demand"][:120], label="daily demand", title="Sample sales · first 120 days",
               ylabel="units", xlabel="day")
    rlhud.plot(hist["promo"][:120] * hist["demand"][:120].max(), label="promo days",
               title="Sample sales · first 120 days")
    prof = weekday_profile(hist)
    rlhud.bars(prof, labels=DOW_NAMES, title="Weekday profile (non-promo days)",
               highlight=int(np.argmax(prof)))
    rlhud.stat("Days of history", n)
    rlhud.stat("Mean demand / day", float(hist["demand"].mean()))
    rlhud.stat("Promo lift", promo_lift(hist), unit="×")

    env = InventoryEnv()
    s, S, train_profit, table = grid_search_sS(env, S_GRID_LOW, S_GRID_HIGH, TRAIN_SEEDS)
    heat = [[table.get((lo, hi), None) for hi in S_GRID_HIGH] for lo in S_GRID_LOW]
    rlhud.grid(heat, title="Mean training profit by (s rows, S columns)", fmt=".0f", cmap="heat",
               labels=[[f"{v / 1000:.1f}k" if v is not None else "—" for v in row] for row in heat])
    test_profit = evaluate_policy(env, lambda e: s_S_policy(e, s, S), seeds=TEST_SEEDS)
    G, totals, trace = run_policy(env, s_S_policy(env, s, S), seed=100, record=True)
    rlhud.plot(trace["demand"][:56], label="demand", title="Tuned (s, S) · first 8 weeks of a test year")
    rlhud.plot(trace["inventory"][:56], label="stock after sales", title="Tuned (s, S) · first 8 weeks of a test year")
    rlhud.stat("Tuned (s, S)", f"({s}, {S})")
    rlhud.stat("Profit · train years", train_profit)
    rlhud.stat("Profit · held-out test years", test_profit, good=test_profit >= 12500)
    rlhud.stat("Fill rate (test year 100)", totals["sales"] / totals["demand"])

In [ ]:
#@title ✅ **Check** · Know your demand, then beat a planner (+250 XP) — tests the code you last ran above
rlcolab.check("15-bonus-inventory/01-baseline-planner")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

Boolean masks do all the work: `normal = promo == 0`, then `d[normal & (dow == k)].mean()` for each weekday k.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

For the lift build `base = np.array([d[normal & (dow == k)].mean() for k in range(7)])`; the expected demand on the promo days is `base[dow[promo == 1]].sum()`.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

Python closures capture variables, not values — but evaluate_policy calls your lambda immediately, so `lambda e: s_S_policy(e, s, S)` inside the loop is fine. Keep `best = (None, None, -np.inf)` and update it whenever a profit is larger.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("15-bonus-inventory/01-baseline-planner")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 15-bonus-inventory/01-baseline-planner
# Reference solution
import numpy as np
import rlhud
from inventory import (InventoryEnv, load_demand_csv, sample_csv_path, s_S_policy,
                       evaluate_policy, run_policy, DOW_NAMES)


def weekday_profile(history):
    """Mean demand per weekday divided by the mean over all days, using NON-promo days only.

    history: dict with numpy arrays "demand", "dow" (0 = Mon) and "promo" (1 on promo days).
    Returns a length-7 array that averages ~1.0 (Saturday ≈ 2.0 means twice a normal day)."""
    d, dow, promo = (np.asarray(history[k]) for k in ("demand", "dow", "promo"))
    normal = promo == 0
    overall = d[normal].mean()
    return np.array([d[normal & (dow == k)].mean() / overall for k in range(7)])


def promo_lift(history):
    """How much a promotion multiplies demand, corrected for the weekday mix.

    lift = (total demand on promo days) / (total demand you would EXPECT on those days
    without a promotion), where the expectation for a day is the mean non-promo demand
    of its weekday."""
    d, dow, promo = (np.asarray(history[k]) for k in ("demand", "dow", "promo"))
    normal = promo == 0
    base = np.array([d[normal & (dow == k)].mean() for k in range(7)])
    on = promo == 1
    return float(d[on].sum() / base[dow[on]].sum())


def grid_search_sS(env, s_grid, S_grid, seeds):
    """Try every (s, S) with S > s; score = mean episode profit over `seeds`
    (use inventory.evaluate_policy with inventory.s_S_policy).

    Returns (best_s, best_S, best_profit, table) where table[(s, S)] = mean profit."""
    table = {}
    best = (None, None, -np.inf)
    for s in s_grid:
        for S in S_grid:
            if S <= s:
                continue
            profit = evaluate_policy(env, lambda e: s_S_policy(e, s, S), seeds=seeds)
            table[(s, S)] = profit
            if profit > best[2]:
                best = (s, S, profit)
    return best[0], best[1], best[2], table


TRAIN_SEEDS = range(0, 4)        # demand years we tune on
TEST_SEEDS = range(100, 120)     # demand years we never looked at while tuning
S_GRID_LOW = range(20, 81, 10)   # reorder points s
S_GRID_HIGH = range(30, 121, 10)  # order-up-to levels S


if __name__ == "__main__":
    hist = load_demand_csv(sample_csv_path())
    n = len(hist["demand"])
    rlhud.plot(hist["demand"][:120], label="daily demand", title="Sample sales · first 120 days",
               ylabel="units", xlabel="day")
    rlhud.plot(hist["promo"][:120] * hist["demand"][:120].max(), label="promo days",
               title="Sample sales · first 120 days")
    prof = weekday_profile(hist)
    rlhud.bars(prof, labels=DOW_NAMES, title="Weekday profile (non-promo days)",
               highlight=int(np.argmax(prof)))
    rlhud.stat("Days of history", n)
    rlhud.stat("Mean demand / day", float(hist["demand"].mean()))
    rlhud.stat("Promo lift", promo_lift(hist), unit="×")

    env = InventoryEnv()
    s, S, train_profit, table = grid_search_sS(env, S_GRID_LOW, S_GRID_HIGH, TRAIN_SEEDS)
    heat = [[table.get((lo, hi), None) for hi in S_GRID_HIGH] for lo in S_GRID_LOW]
    rlhud.grid(heat, title="Mean training profit by (s rows, S columns)", fmt=".0f", cmap="heat",
               labels=[[f"{v / 1000:.1f}k" if v is not None else "—" for v in row] for row in heat])
    test_profit = evaluate_policy(env, lambda e: s_S_policy(e, s, S), seeds=TEST_SEEDS)
    G, totals, trace = run_policy(env, s_S_policy(env, s, S), seed=100, record=True)
    rlhud.plot(trace["demand"][:56], label="demand", title="Tuned (s, S) · first 8 weeks of a test year")
    rlhud.plot(trace["inventory"][:56], label="stock after sales", title="Tuned (s, S) · first 8 weeks of a test year")
    rlhud.stat("Tuned (s, S)", f"({s}, {S})")
    rlhud.stat("Profit · train years", train_profit)
    rlhud.stat("Profit · held-out test years", test_profit, good=test_profit >= 12500)
    rlhud.stat("Fill rate (test year 100)", totals["sales"] / totals["demand"])

## 04 · Reward design: you get what you pay for

The reward is "just profit", yet there are several ways to get it badly wrong.

> **⚠️ WARNING** · **The agent that never stocks anything**
>
> Charge the purchase price when goods are ordered, end the episode after 365 days, and forget
> to value the leftover stock: near the end of every episode ordering looks like pure loss, and
> with short training horizons the agent learns that *empty shelves are cheap*. The simulator
> credits leftover stock at cost on the final day (`salvage=True`) to remove that cliff.

**Cash vs accrual.** Paying for goods the day you order them (`accounting="cash"`) and booking a
unit's cost when it is sold or spoils (`accounting="accrual"`, the default) give the same yearly
profit, up to the value of the starting stock. They are very different learning problems: with
cash accounting an order is a big negative reward followed by small positive ones up to a week
later; with accrual, each day's reward reflects mostly that day's decisions. Same objective,
much easier credit assignment.

**Missing costs get exploited.** Set the stockout penalty and spoilage to zero and the agent
happily runs the shelf empty every Sunday. Set holding cost too high and it learns the same thing
for a different reason. Every number in the reward is a business judgement — agree on them with
the people who own the P&L, not by tuning until the curve looks nice.

**Discounting vs quarterly targets.** γ = 0.95 means a profit 14 days out counts half. That is a
statement about *credit assignment*, not about what the business values. If management is paid on
quarterly profit, the true objective is an undiscounted 90-day sum; γ < 1 is a tool to make learning
stable, and you should still **evaluate** on the undiscounted business metric.

In [ ]:
#@title 🎮 **Interactive** · Discounting explorer
rlcolab.widget("discount", gamma=0.95, rewards=[-5, 0, 24, 24, 24, -3, -3, -3, -3, -3, -3, -3, -3, -3])

An order: a fee today, margins when it sells, then holding and spoilage costs if you over-ordered.
Lower γ and the late costs nearly vanish — the agent becomes blind to over-stocking.

In [ ]:
#@title 🧠 **Knowledge check 2** · +25 XP — run, then pick an answer
rlcolab.quiz("15-bonus-inventory#1", {"q": "Your agent's test profit is great, but you notice it lets the shelf run empty most Sundays. What should you check first?", "options": ["Increase the learning rate", "Whether the stockout penalty and lost margin in the reward reflect what an empty shelf really costs the business", "Use a deeper neural network"], "answer": 1, "explain": "The agent maximised exactly what you rewarded. If an empty shelf costs only the lost margin of a few units, being out of stock on a quiet day is optimal. If customers who find an empty shelf go elsewhere for good, the reward must say so."})

## 05 · Learning on top of the planner

Plain tabular Q-learning on the full state can learn this problem — eventually. In a budget of a
few hundred simulated years it does **not** reliably match a well-tuned (s, S). Rewards are noisy
(demand varies ±40 % day to day), most of the state space is rarely visited, and the greedy policy
inherits the noise in every Q-value.

The practical answer is **residual RL**: keep the planner, and let the agent learn *adjustments*.
Here the agent sees only a 14-value context (weekday × promotion coming) and chooses an offset that
shifts both s and S for today. Offset 0 *is* the planner, so an untrained agent is exactly as good
as today's process, and every gain is something the data supports: stock up on Thursday for the
Saturday rush, lean out on Sunday, prepare for a promotion.

> **💡 TIP** · **Why industry loves residual policies**
>
> They are safe to deploy (the fallback is the existing process), easy to explain to a planner
> ("on Thursdays the agent orders up to 20 more"), and they need far less data than learning
> from scratch. The same idea appears as residual control in robotics and as "RL on top of a
> heuristic" in pricing and logistics.

---
### 🧪 Exercise 2 · Residual Q-learning: teach the planner new tricks · ⭐ **300 XP**

Plain Q-learning from scratch struggles to even *match* a tuned (s, S) policy within a few hundred simulated years — the demo shows it. Industry's answer: **keep the planner and let RL learn adjustments**.

The agent sees a tiny **context** — weekday × *promo soon* (14 states) — and picks an **offset** that shifts both s and S for today. Offset 0 *is* the planner, so the agent can only improve on it where the data says so (e.g. stock up before Saturdays and promotions).

1. `context(env, obs)` → `dow * 2 + promo_soon`, using `env.decode_state(obs)` which returns `(position_bucket, dow, promo_soon)`.
2. `adjusted_action(env, s, S, offset)` → order nothing while `env.position > s + offset`, otherwise order up to `S + offset` via `env.nearest_action(units)`.
3. `train_residual_q(env, s, S, offsets, episodes, alpha, gamma, eps_start, seed)` → `(Q, returns)`: tabular Q-learning over `(context, offset index)`. Q starts at 0 with **+1.0 on the zero-offset column** so the untrained agent equals the planner. ε decays linearly from `eps_start` to 0 over the first 70% of episodes. Reset every episode with a fresh seed drawn from your `rng`.
4. `residual_policy(env, Q, s, S, offsets)` → greedy `obs -> action`.

Rewards are daily profits, reported with **accrual accounting** (a unit's cost is booked when it is sold or spoils), so a good order is rewarded within days.

**Pass:** unit checks on hand-built states, and the agent beats the tuned planner (s=50, S=70) by **≥ 2 %** profit on 20 held-out simulated years, averaged over 3 training seeds.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 15-bonus-inventory/02-residual-q
import numpy as np
import rlhud
from inventory import InventoryEnv, s_S_policy, run_policy, evaluate_policy, DOW_NAMES

# The planner's tuned policy from exercise 1 — RL learns *adjustments* to it.
BASE_s, BASE_S = 50, 70
OFFSETS = np.array([-20, -10, 0, 10, 20, 30, 40])   # shift both s and S by this many units
N_CONTEXTS = 14                                      # 7 weekdays × promo-soon flag


def context(env, obs):
    """Map the env's tabular observation to a small context index in [0, 14):
    dow * 2 + promo_soon. Use env.decode_state(obs) -> (position_bucket, dow, promo_soon).
    The position is deliberately dropped: the (s, S) rule already reacts to it."""
    # TODO
    raise NotImplementedError


def adjusted_action(env, s, S, offset):
    """The env action of an (s + offset, S + offset) policy in the env's current state:
    order nothing while env.position > s + offset, otherwise order up to S + offset
    (env.nearest_action(units) converts a number of units into an action index)."""
    # TODO
    raise NotImplementedError


def train_residual_q(env, s=BASE_s, S=BASE_S, offsets=OFFSETS, episodes=500, alpha=0.05,
                     gamma=0.95, eps_start=0.5, seed=0):
    """Tabular Q-learning over (context, offset). Returns (Q, returns).

    - Q has shape (N_CONTEXTS, len(offsets)); start at 0 but give the zero-offset column a
      small head start (+1.0) so that, before learning anything, the agent IS the planner.
    - ε decays linearly from eps_start to 0 over the first 70% of the episodes.
    - Each episode: env.reset(seed=<a fresh int from rng>), then step until truncated."""
    rng = np.random.default_rng(seed)
    offsets = np.asarray(offsets)
    Q = np.zeros((N_CONTEXTS, len(offsets)))
    # TODO: head start for the zero-offset column
    returns = []
    for ep in range(episodes):
        # TODO: ε schedule, reset with a fresh seed, run one episode:
        #   k = ε-greedy offset index for the current context
        #   step with adjusted_action(env, s, S, offsets[k])
        #   Q-learning update on Q[c, k]; accumulate the episode's profit
        raise NotImplementedError
    return Q, returns


def residual_policy(env, Q, s=BASE_s, S=BASE_S, offsets=OFFSETS):
    """Greedy policy obs -> action: pick the best offset for the context, apply it."""
    offsets = np.asarray(offsets)
    # TODO: return a function obs -> action
    raise NotImplementedError


def train_plain_q(env, episodes=300, alpha=0.1, gamma=0.9, seed=0):
    """For comparison: Q-learning from scratch on the full tabular state (position too)."""
    rng = np.random.default_rng(seed)
    Q = np.full((env.n_states, env.n_actions), 400.0)        # optimistic start
    returns = []
    for ep in range(episodes):
        eps = max(0.02, 1 - ep / (0.7 * episodes))
        s, _ = env.reset(seed=int(rng.integers(2**31)))
        G, done = 0.0, False
        while not done:
            a = int(rng.integers(env.n_actions)) if rng.random() < eps else int(np.argmax(Q[s]))
            s2, r, terminated, truncated, _ = env.step(a)
            Q[s, a] += alpha * (r + gamma * Q[s2].max() - Q[s, a])
            s, G, done = s2, G + r, terminated or truncated
        returns.append(G)
    return Q, returns


TEST_SEEDS = range(100, 120)

if __name__ == "__main__":
    env = InventoryEnv()
    Q, returns = train_residual_q(env, seed=0)
    Qp, returns_plain = train_plain_q(env, seed=0)
    rlhud.plot(returns, label="residual Q (offsets on (s, S))", title="Training profit per simulated year", smooth=20)
    rlhud.plot(returns_plain, label="plain Q from scratch", title="Training profit per simulated year", smooth=20)

    base = evaluate_policy(env, lambda e: s_S_policy(e, BASE_s, BASE_S), seeds=TEST_SEEDS)
    agent = evaluate_policy(env, lambda e: residual_policy(e, Q), seeds=TEST_SEEDS)
    plain = evaluate_policy(env, lambda e: (lambda o: int(np.argmax(Qp[o]))), seeds=TEST_SEEDS)
    rlhud.stat("Planner (s, S) · test profit", base)
    rlhud.stat("Plain Q · test profit", plain, good=plain > base)
    rlhud.stat("Residual Q · test profit", agent, good=agent > base)
    rlhud.stat("Uplift vs planner", 100 * (agent / base - 1), unit=" %", good=agent > base)

    best = [int(OFFSETS[np.argmax(Q[c])]) for c in range(N_CONTEXTS)]
    rlhud.bars(best, labels=[f"{DOW_NAMES[c // 2]}{'+P' if c % 2 else ''}" for c in range(N_CONTEXTS)],
               title="Learned shift of (s, S) per context (+P = promo soon)", ylabel="units")

    _, tb, trb = run_policy(env, s_S_policy(env, BASE_s, BASE_S), seed=100, record=True)
    _, ta, tra = run_policy(env, residual_policy(env, Q), seed=100, record=True)
    days = slice(140, 196)
    title = "Test year 100 · 8 weeks: demand vs stock after sales"
    rlhud.plot(tra["demand"][days], label="demand", title=title, xlabel="day")
    rlhud.plot(trb["inventory"][days], label="planner stock", title=title)
    rlhud.plot(tra["inventory"][days], label="agent stock", title=title)

    rows = []
    for key in ("revenue", "cogs", "waste", "order_fee", "holding", "stockout"):
        rows.append([key, round(trb["costs"][key]), round(tra["costs"][key])])
    rows += [["lost sales (units)", tb["lost"], ta["lost"]],
             ["spoiled (units)", tb["spoiled"], ta["spoiled"]],
             ["orders placed", tb["orders"], ta["orders"]],
             ["fill rate", round(tb["sales"] / tb["demand"], 3), round(ta["sales"] / ta["demand"], 3)],
             ["profit", round(tb["profit"]), round(ta["profit"])]]
    rlhud.table(rows, headers=["test year 100", "planner (s, S)", "residual Q"], title="Cost breakdown")

In [ ]:
#@title ✅ **Check** · Residual Q-learning: teach the planner new tricks (+300 XP) — tests the code you last ran above
rlcolab.check("15-bonus-inventory/02-residual-q")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

`_, dow, promo_soon = env.decode_state(obs)`; for the action: `if env.position > s + offset: return 0` else `return env.nearest_action(S + offset - env.position)`.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

Head start: `Q[:, int(np.flatnonzero(offsets == 0)[0])] = 1.0`. ε: `eps = max(0.0, eps_start * (1 - ep / (0.7 * episodes)))`.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

Inside the episode: `k = int(rng.integers(len(offsets))) if rng.random() < eps else int(np.argmax(Q[c]))`, step with `adjusted_action(env, s, S, offsets[k])`, then `Q[c, k] += alpha * (r + gamma * Q[c2].max() - Q[c, k])` where c2 = context of the new obs. The episode ends when `truncated` is True.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("15-bonus-inventory/02-residual-q")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 15-bonus-inventory/02-residual-q
# Reference solution
import numpy as np
import rlhud
from inventory import InventoryEnv, s_S_policy, run_policy, evaluate_policy, DOW_NAMES

# The planner's tuned policy from exercise 1 — RL learns *adjustments* to it.
BASE_s, BASE_S = 50, 70
OFFSETS = np.array([-20, -10, 0, 10, 20, 30, 40])   # shift both s and S by this many units
N_CONTEXTS = 14                                      # 7 weekdays × promo-soon flag


def context(env, obs):
    """Map the env's tabular observation to a small context index in [0, 14):
    dow * 2 + promo_soon. Use env.decode_state(obs) -> (position_bucket, dow, promo_soon).
    The position is deliberately dropped: the (s, S) rule already reacts to it."""
    _, dow, promo_soon = env.decode_state(obs)
    return dow * 2 + promo_soon


def adjusted_action(env, s, S, offset):
    """The env action of an (s + offset, S + offset) policy in the env's current state:
    order nothing while env.position > s + offset, otherwise order up to S + offset
    (env.nearest_action(units) converts a number of units into an action index)."""
    pos = env.position
    if pos > s + offset:
        return 0
    return env.nearest_action(S + offset - pos)


def train_residual_q(env, s=BASE_s, S=BASE_S, offsets=OFFSETS, episodes=500, alpha=0.05,
                     gamma=0.95, eps_start=0.5, seed=0):
    """Tabular Q-learning over (context, offset). Returns (Q, returns).

    - Q has shape (N_CONTEXTS, len(offsets)); start at 0 but give the zero-offset column a
      small head start (+1.0) so that, before learning anything, the agent IS the planner.
    - ε decays linearly from eps_start to 0 over the first 70% of the episodes.
    - Each episode: env.reset(seed=<a fresh int from rng>), then step until truncated."""
    rng = np.random.default_rng(seed)
    offsets = np.asarray(offsets)
    Q = np.zeros((N_CONTEXTS, len(offsets)))
    Q[:, int(np.flatnonzero(offsets == 0)[0])] = 1.0
    returns = []
    for ep in range(episodes):
        eps = max(0.0, eps_start * (1 - ep / (0.7 * episodes)))
        obs, _ = env.reset(seed=int(rng.integers(2**31)))
        c = context(env, obs)
        G, done = 0.0, False
        while not done:
            k = int(rng.integers(len(offsets))) if rng.random() < eps else int(np.argmax(Q[c]))
            obs, r, terminated, truncated, _ = env.step(adjusted_action(env, s, S, offsets[k]))
            c2 = context(env, obs)
            target = r if terminated else r + gamma * Q[c2].max()
            Q[c, k] += alpha * (target - Q[c, k])
            c, G, done = c2, G + r, terminated or truncated
        returns.append(G)
    return Q, returns


def residual_policy(env, Q, s=BASE_s, S=BASE_S, offsets=OFFSETS):
    """Greedy policy obs -> action: pick the best offset for the context, apply it."""
    offsets = np.asarray(offsets)
    return lambda obs: adjusted_action(env, s, S, offsets[int(np.argmax(Q[context(env, obs)]))])


def train_plain_q(env, episodes=300, alpha=0.1, gamma=0.9, seed=0):
    """For comparison: Q-learning from scratch on the full tabular state (position too)."""
    rng = np.random.default_rng(seed)
    Q = np.full((env.n_states, env.n_actions), 400.0)        # optimistic start
    returns = []
    for ep in range(episodes):
        eps = max(0.02, 1 - ep / (0.7 * episodes))
        s, _ = env.reset(seed=int(rng.integers(2**31)))
        G, done = 0.0, False
        while not done:
            a = int(rng.integers(env.n_actions)) if rng.random() < eps else int(np.argmax(Q[s]))
            s2, r, terminated, truncated, _ = env.step(a)
            Q[s, a] += alpha * (r + gamma * Q[s2].max() - Q[s, a])
            s, G, done = s2, G + r, terminated or truncated
        returns.append(G)
    return Q, returns


TEST_SEEDS = range(100, 120)

if __name__ == "__main__":
    env = InventoryEnv()
    Q, returns = train_residual_q(env, seed=0)
    Qp, returns_plain = train_plain_q(env, seed=0)
    rlhud.plot(returns, label="residual Q (offsets on (s, S))", title="Training profit per simulated year", smooth=20)
    rlhud.plot(returns_plain, label="plain Q from scratch", title="Training profit per simulated year", smooth=20)

    base = evaluate_policy(env, lambda e: s_S_policy(e, BASE_s, BASE_S), seeds=TEST_SEEDS)
    agent = evaluate_policy(env, lambda e: residual_policy(e, Q), seeds=TEST_SEEDS)
    plain = evaluate_policy(env, lambda e: (lambda o: int(np.argmax(Qp[o]))), seeds=TEST_SEEDS)
    rlhud.stat("Planner (s, S) · test profit", base)
    rlhud.stat("Plain Q · test profit", plain, good=plain > base)
    rlhud.stat("Residual Q · test profit", agent, good=agent > base)
    rlhud.stat("Uplift vs planner", 100 * (agent / base - 1), unit=" %", good=agent > base)

    best = [int(OFFSETS[np.argmax(Q[c])]) for c in range(N_CONTEXTS)]
    rlhud.bars(best, labels=[f"{DOW_NAMES[c // 2]}{'+P' if c % 2 else ''}" for c in range(N_CONTEXTS)],
               title="Learned shift of (s, S) per context (+P = promo soon)", ylabel="units")

    _, tb, trb = run_policy(env, s_S_policy(env, BASE_s, BASE_S), seed=100, record=True)
    _, ta, tra = run_policy(env, residual_policy(env, Q), seed=100, record=True)
    days = slice(140, 196)
    title = "Test year 100 · 8 weeks: demand vs stock after sales"
    rlhud.plot(tra["demand"][days], label="demand", title=title, xlabel="day")
    rlhud.plot(trb["inventory"][days], label="planner stock", title=title)
    rlhud.plot(tra["inventory"][days], label="agent stock", title=title)

    rows = []
    for key in ("revenue", "cogs", "waste", "order_fee", "holding", "stockout"):
        rows.append([key, round(trb["costs"][key]), round(tra["costs"][key])])
    rows += [["lost sales (units)", tb["lost"], ta["lost"]],
             ["spoiled (units)", tb["spoiled"], ta["spoiled"]],
             ["orders placed", tb["orders"], ta["orders"]],
             ["fill rate", round(tb["sales"] / tb["demand"], 3), round(ta["sales"] / ta["demand"], 3)],
             ["profit", round(tb["profit"]), round(ta["profit"])]]
    rlhud.table(rows, headers=["test year 100", "planner (s, S)", "residual Q"], title="Cost breakdown")

## 06 · Data: simulated, then real

The bundled file `data/sample_sales.csv` holds two years of daily sales for one product, generated
with weekly seasonality, a yearly cycle, a slow upward trend, a pre-Christmas rush and planned
promotions (columns `date,demand,dow,promo`).

`load_demand_csv(path)` needs only `date` and a quantity column; `dow` is recomputed and `promo`
is optional. `InventoryEnv(demand_series=..., promo_series=..., dates=...)` then **replays** that
history day by day, and `jitter_weeks=k` adds a seasonal bootstrap (each day's demand drawn from the
same weekday up to *k* weeks away) so you can train on one historical year without memorising it.

**Bring your own data.** Any daily sales export works. For example, the Kaggle *Store Sales — Time
Series Forecasting* data (Favorita) has `date, store_nbr, family, sales, onpromotion`: keep one store
and one product family, then

```python
import csv
rows = [r for r in csv.DictReader(open("train.csv")) if r["store_nbr"] == "1" and r["family"] == "BREAD/BAKERY"]
with open("my_sales.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["date", "demand", "promo"])
    for r in rows:
        w.writerow([r["date"], r["sales"], int(float(r["onpromotion"]) > 0)])
```

Order sizes, capacity and the search grids scale with the product's mean demand, so the same code
runs on a product that sells 3 a day or 300.

> **⚠️ WARNING** · **Never let the future leak into training**
>
> Split time series **by time**: train on the past, test on the future. A random split lets the
> model "see" next week's promotion effects while learning about this week. The same goes for
> every number derived from data — the mean demand that scales order sizes, the tuned (s, S), the
> weekday profile — compute them on the training period only.

The honest caveats of offline evaluation:

- **Sales are not demand.** On days you ran out, the history records what you *sold*, not what
  customers *wanted*. Replaying sales as demand underestimates it exactly when it matters. Real
  projects "uncensor" demand from stockout flags or comparable stores.
- **Replay assumes your actions do not change demand.** Here they do not; in reality, an empty
  shelf trains customers to shop elsewhere and a full display sells more.
- **Sim-to-real gap.** Real lead times vary, suppliers short-ship, staff mis-count. An agent that
  exploits a quirk of the simulator will fail in the store. Stress-test it on perturbed
  simulators (longer lead times, noisier demand) before trusting it.
- **One test year is one sample.** A 3 % uplift on a single year may be luck. Look at several
  products, several stores, and at *where* the gain comes from (fill rate? fewer orders? less waste?).

In [ ]:
#@title 🧠 **Knowledge check 3** · +25 XP — run, then pick an answer
rlcolab.quiz("15-bonus-inventory#2", {"q": "You scale the test environment with the mean demand of the TEST year instead of the training year. Why is that a problem?", "options": ["It isn't — scaling is not learning", "It leaks information about the future (e.g. the year-on-year growth) into the policy, so the test result is optimistic", "It makes the simulation slower"], "answer": 1, "explain": "Order sizes and capacity would silently adapt to next year's demand level — something you could never know in advance. Any statistic computed on test data is a leak."})

---
### 🧪 Exercise 3 · Going real: an out-of-time evaluation · ⭐ **300 XP**

Time to treat this like a real project. The bundled file `data/sample_sales.csv` holds **two years** of daily sales (columns `date,demand,dow,promo`). You will **train on year 1** and **judge on year 2** — exactly what you would do before showing results to a store manager.

The planner tuning and the residual Q-learner from exercises 1–2 are provided at the bottom of the file. You write the harness around them:

1. `time_split(history, n_train)` — first `n_train` days → train, the rest → test, for every column. Never shuffle a time series.
2. `fill_rate(sales, demand)` — total sales ÷ total demand (1.0 if there was no demand).
3. `summarize(totals)` — `{"profit", "fill_rate", "lost_sales", "spoiled", "orders"}` from the `totals` dict that `run_policy` returns (keys `profit, demand, sales, lost, spoiled, overflow, ordered, orders`).
4. `make_envs(train, test)` — a **training** env that replays year 1 with a seasonal bootstrap (`jitter_weeks=2`), and a **test** env that replays year 2 exactly, scaled with `ref_demand=` the *training* mean.
5. `evaluate(env, make_policy, seeds)` — mean of each `summarize` metric over one `run_policy` episode per seed.
6. `run_pipeline(csv_path, n_train, seed)` — load, split, build envs, tune (s, S) on the train env, train the agent on the train env, evaluate both on the test env. Return `{"s_S", "baseline", "agent", "Q"}`.

**Your own data:** export one product's daily sales as CSV with at least `date` (YYYY-MM-DD) and a quantity column, e.g. one store × one item of a Kaggle store-sales dataset. Put it in your workspace and set `CSV_PATH = "my_sales.csv"` (if the quantity column is called `sales`, use `load_demand_csv(path, demand_col="sales")`). Order sizes, capacity and the search grid scale to your product automatically.

**Pass:** split, metrics and envs pass hand-made cases; the planner is tuned on year 1 only; averaged over 2 training seeds, the agent earns **at least as much as the planner on year 2** with a fill rate **≥ 88 %**.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 15-bonus-inventory/03-real-data
import numpy as np
import rlhud
from inventory import (InventoryEnv, load_demand_csv, sample_csv_path, s_S_policy,
                       evaluate_policy, run_policy, DOW_NAMES)

# Point this at your own export (columns date,demand[,promo]) to run the whole study on it.
CSV_PATH = sample_csv_path()


# ============================================================ your work: the evaluation harness
def time_split(history, n_train):
    """Split a history dict by TIME: the first n_train days train, everything after tests.
    Every key ("dates", "demand", "dow", "promo") is sliced the same way. Never shuffle!"""
    # TODO
    raise NotImplementedError


def fill_rate(sales, demand):
    """Fraction of demand that was served: total sales / total demand (1.0 if no demand).
    Accepts numbers or sequences."""
    # TODO
    raise NotImplementedError


def summarize(totals):
    """Business metrics from env.totals (as returned by run_policy):
    {"profit", "fill_rate", "lost_sales", "spoiled", "orders"}."""
    # TODO: totals has keys profit, demand, sales, lost, spoiled, overflow, ordered, orders
    raise NotImplementedError


def make_envs(train, test):
    """Build (train_env, test_env) from the two halves of the history.

    train_env: replays the training days with a seasonal bootstrap (jitter_weeks=2) so every
               episode is a slightly different but realistic year; horizon = len(train days).
    test_env : replays the test days exactly as they happened (no jitter, no random start),
               horizon = len(test days). Its scale settings (order sizes, capacity) must come
               from the TRAINING data: pass ref_demand = mean training demand."""
    # TODO: InventoryEnv(demand_series=..., promo_series=..., dates=..., horizon=..., ...)
    raise NotImplementedError


def evaluate(env, make_policy, seeds=range(5)):
    """Run make_policy(env) for one episode per seed (inventory.run_policy) and return the
    mean of each summarize() metric over the seeds."""
    # TODO: run_policy(env, policy, seed=s) returns (total_reward, totals, trace)
    raise NotImplementedError


def run_pipeline(csv_path=CSV_PATH, n_train=365, seed=0):
    """The full study. Returns {"s_S": (s, S), "baseline": metrics, "agent": metrics, "Q": Q}.

    1. load the CSV, time_split it, make_envs
    2. tune the planner on the TRAIN env: s, S, _, _ = grid_search_sS(train_env)
       (the search grid scales itself to the product's order size)
    3. train the residual agent on the TRAIN env: train_residual_q(train_env, s, S, seed=seed)
    4. evaluate both on the TEST env with evaluate(...) and the default seeds"""
    hist = load_demand_csv(csv_path)
    # TODO: steps 1-4
    raise NotImplementedError


# ============================================================ from exercises 1 and 2 (done)
TUNE_SEEDS = range(4)
OFFSETS = np.array([-20, -10, 0, 10, 20, 30, 40])
N_CONTEXTS = 14


def _grid(env):
    step = env.order_step       # scale the search grid to this product's order size
    return range(4 * step, 15 * step, 2 * step), range(5 * step, 21 * step, 2 * step)


def grid_search_sS(env, s_grid=None, S_grid=None, seeds=TUNE_SEEDS):
    if s_grid is None or S_grid is None:
        s_grid, S_grid = _grid(env)
    table, best = {}, (None, None, -np.inf)
    for s in s_grid:
        for S in S_grid:
            if S > s:
                table[(s, S)] = p = evaluate_policy(env, lambda e: s_S_policy(e, s, S), seeds=seeds)
                if p > best[2]:
                    best = (s, S, p)
    return best[0], best[1], best[2], table


def context(env, obs):
    _, dow, promo_soon = env.decode_state(obs)
    return dow * 2 + promo_soon


def adjusted_action(env, s, S, offset):
    pos = env.position
    return 0 if pos > s + offset else env.nearest_action(S + offset - pos)


def train_residual_q(env, s, S, offsets=OFFSETS, episodes=500, alpha=0.05, gamma=0.95,
                     eps_start=0.5, seed=0):
    rng = np.random.default_rng(seed)
    offsets = np.asarray(offsets) * env.order_step / 6     # offsets were tuned for 6-unit steps
    Q = np.zeros((N_CONTEXTS, len(offsets)))
    Q[:, int(np.flatnonzero(offsets == 0)[0])] = 1.0
    returns = []
    for ep in range(episodes):
        eps = max(0.0, eps_start * (1 - ep / (0.7 * episodes)))
        obs, _ = env.reset(seed=int(rng.integers(2**31)))
        c, G, done = context(env, obs), 0.0, False
        while not done:
            k = int(rng.integers(len(offsets))) if rng.random() < eps else int(np.argmax(Q[c]))
            obs, r, terminated, truncated, _ = env.step(adjusted_action(env, s, S, offsets[k]))
            c2 = context(env, obs)
            Q[c, k] += alpha * ((r if terminated else r + gamma * Q[c2].max()) - Q[c, k])
            c, G, done = c2, G + r, terminated or truncated
        returns.append(G)
    return Q, returns


def residual_policy(env, Q, s, S, offsets=OFFSETS):
    offsets = np.asarray(offsets) * env.order_step / 6
    return lambda obs: adjusted_action(env, s, S, offsets[int(np.argmax(Q[context(env, obs)]))])


if __name__ == "__main__":
    hist = load_demand_csv(CSV_PATH)
    n_train = 365
    rlhud.plot(hist["demand"], label="daily demand", title="Full history · train | test split at day 365",
               smooth=7, xlabel="day")
    res = run_pipeline(CSV_PATH, n_train=n_train)
    s, S = res["s_S"]
    rlhud.stat("Tuned on year 1: (s, S)", f"({s}, {S})")
    rows = []
    for k in ("profit", "fill_rate", "lost_sales", "spoiled", "orders"):
        b, a = res["baseline"][k], res["agent"][k]
        rows.append([k, round(b, 3), round(a, 3), f"{100 * (a / b - 1):+.1f} %" if b else "—"])
    rlhud.table(rows, headers=["test year (unseen)", "planner (s, S)", "residual Q", "change"],
                title="Out-of-time evaluation")
    up = res["agent"]["profit"] / res["baseline"]["profit"] - 1
    rlhud.stat("Profit uplift on the test year", 100 * up, unit=" %", good=up > 0)
    rlhud.stat("Agent fill rate", res["agent"]["fill_rate"], good=res["agent"]["fill_rate"] >= 0.9)

    # what does the agent do differently? learned (s, S) shift per context
    Q = res["Q"]
    step = InventoryEnv(demand_series=hist["demand"][:n_train]).order_step
    shift = [float(OFFSETS[np.argmax(Q[c])] * step / 6) for c in range(N_CONTEXTS)]
    rlhud.bars(shift, labels=[f"{DOW_NAMES[c // 2]}{'+P' if c % 2 else ''}" for c in range(N_CONTEXTS)],
               title="Learned shift of (s, S) in units, per context (+P = promo soon)")
    if up <= 0:
        rlhud.text("On this data the agent did not beat the planner out of time. That is a real, "
                   "useful result: check the train/test demand shift and the size of the uplift "
                   "on the training year before trusting any RL policy.", title="Honest result", tone="warn")

In [ ]:
#@title ✅ **Check** · Going real: an out-of-time evaluation (+300 XP) — tests the code you last ran above
rlcolab.check("15-bonus-inventory/03-real-data")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

time_split: `{k: v[:n_train] for k, v in history.items()}` and the same with `v[n_train:]`. Lists and numpy arrays both slice this way.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

make_envs: `InventoryEnv(demand_series=train['demand'], promo_series=train['promo'], dates=train['dates'], horizon=len(train['demand']), jitter_weeks=2)`; the test env drops jitter and adds `ref_demand=float(np.mean(train['demand']))`.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

evaluate: `rows = [summarize(run_policy(env, make_policy(env), seed=s)[1]) for s in seeds]`, then average each key. run_pipeline: `s, S, _, _ = grid_search_sS(train_env)`, `Q, _ = train_residual_q(train_env, s, S, seed=seed)`, and `evaluate(test_env, lambda e: residual_policy(e, Q, s, S))`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("15-bonus-inventory/03-real-data")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 15-bonus-inventory/03-real-data
# Reference solution
import numpy as np
import rlhud
from inventory import (InventoryEnv, load_demand_csv, sample_csv_path, s_S_policy,
                       evaluate_policy, run_policy, DOW_NAMES)

# Point this at your own export (columns date,demand[,promo]) to run the whole study on it.
CSV_PATH = sample_csv_path()


# ============================================================ your work: the evaluation harness
def time_split(history, n_train):
    """Split a history dict by TIME: the first n_train days train, everything after tests.
    Every key ("dates", "demand", "dow", "promo") is sliced the same way. Never shuffle!"""
    train = {k: v[:n_train] for k, v in history.items()}
    test = {k: v[n_train:] for k, v in history.items()}
    return train, test


def fill_rate(sales, demand):
    """Fraction of demand that was served: total sales / total demand (1.0 if no demand).
    Accepts numbers or sequences."""
    total = float(np.sum(demand))
    return 1.0 if total == 0 else float(np.sum(sales)) / total


def summarize(totals):
    """Business metrics from env.totals (as returned by run_policy):
    {"profit", "fill_rate", "lost_sales", "spoiled", "orders"}."""
    return {"profit": float(totals["profit"]),
            "fill_rate": fill_rate(totals["sales"], totals["demand"]),
            "lost_sales": float(totals["lost"]),
            "spoiled": float(totals["spoiled"]),
            "orders": float(totals["orders"])}


def make_envs(train, test):
    """Build (train_env, test_env) from the two halves of the history.

    train_env: replays the training days with a seasonal bootstrap (jitter_weeks=2) so every
               episode is a slightly different but realistic year; horizon = len(train days).
    test_env : replays the test days exactly as they happened (no jitter, no random start),
               horizon = len(test days). Its scale settings (order sizes, capacity) must come
               from the TRAINING data: pass ref_demand = mean training demand."""
    train_env = InventoryEnv(demand_series=train["demand"], promo_series=train["promo"],
                             dates=train["dates"], horizon=len(train["demand"]), jitter_weeks=2)
    test_env = InventoryEnv(demand_series=test["demand"], promo_series=test["promo"],
                            dates=test["dates"], horizon=len(test["demand"]),
                            ref_demand=float(np.mean(train["demand"])))
    return train_env, test_env


def evaluate(env, make_policy, seeds=range(5)):
    """Run make_policy(env) for one episode per seed (inventory.run_policy) and return the
    mean of each summarize() metric over the seeds."""
    rows = [summarize(run_policy(env, make_policy(env), seed=s)[1]) for s in seeds]
    return {k: float(np.mean([r[k] for r in rows])) for k in rows[0]}


def run_pipeline(csv_path=CSV_PATH, n_train=365, seed=0):
    """The full study. Returns {"s_S": (s, S), "baseline": metrics, "agent": metrics, "Q": Q}.

    1. load the CSV, time_split it, make_envs
    2. tune the planner on the TRAIN env: s, S, _, _ = grid_search_sS(train_env)
       (the search grid scales itself to the product's order size)
    3. train the residual agent on the TRAIN env: train_residual_q(train_env, s, S, seed=seed)
    4. evaluate both on the TEST env with evaluate(...) and the default seeds"""
    hist = load_demand_csv(csv_path)
    train, test = time_split(hist, n_train)
    train_env, test_env = make_envs(train, test)
    s, S, _, _ = grid_search_sS(train_env)
    Q, _ = train_residual_q(train_env, s, S, seed=seed)
    return {"s_S": (s, S),
            "baseline": evaluate(test_env, lambda e: s_S_policy(e, s, S)),
            "agent": evaluate(test_env, lambda e: residual_policy(e, Q, s, S)),
            "Q": Q}


# ============================================================ from exercises 1 and 2 (done)
TUNE_SEEDS = range(4)
OFFSETS = np.array([-20, -10, 0, 10, 20, 30, 40])
N_CONTEXTS = 14


def _grid(env):
    step = env.order_step       # scale the search grid to this product's order size
    return range(4 * step, 15 * step, 2 * step), range(5 * step, 21 * step, 2 * step)


def grid_search_sS(env, s_grid=None, S_grid=None, seeds=TUNE_SEEDS):
    if s_grid is None or S_grid is None:
        s_grid, S_grid = _grid(env)
    table, best = {}, (None, None, -np.inf)
    for s in s_grid:
        for S in S_grid:
            if S > s:
                table[(s, S)] = p = evaluate_policy(env, lambda e: s_S_policy(e, s, S), seeds=seeds)
                if p > best[2]:
                    best = (s, S, p)
    return best[0], best[1], best[2], table


def context(env, obs):
    _, dow, promo_soon = env.decode_state(obs)
    return dow * 2 + promo_soon


def adjusted_action(env, s, S, offset):
    pos = env.position
    return 0 if pos > s + offset else env.nearest_action(S + offset - pos)


def train_residual_q(env, s, S, offsets=OFFSETS, episodes=500, alpha=0.05, gamma=0.95,
                     eps_start=0.5, seed=0):
    rng = np.random.default_rng(seed)
    offsets = np.asarray(offsets) * env.order_step / 6     # offsets were tuned for 6-unit steps
    Q = np.zeros((N_CONTEXTS, len(offsets)))
    Q[:, int(np.flatnonzero(offsets == 0)[0])] = 1.0
    returns = []
    for ep in range(episodes):
        eps = max(0.0, eps_start * (1 - ep / (0.7 * episodes)))
        obs, _ = env.reset(seed=int(rng.integers(2**31)))
        c, G, done = context(env, obs), 0.0, False
        while not done:
            k = int(rng.integers(len(offsets))) if rng.random() < eps else int(np.argmax(Q[c]))
            obs, r, terminated, truncated, _ = env.step(adjusted_action(env, s, S, offsets[k]))
            c2 = context(env, obs)
            Q[c, k] += alpha * ((r if terminated else r + gamma * Q[c2].max()) - Q[c, k])
            c, G, done = c2, G + r, terminated or truncated
        returns.append(G)
    return Q, returns


def residual_policy(env, Q, s, S, offsets=OFFSETS):
    offsets = np.asarray(offsets) * env.order_step / 6
    return lambda obs: adjusted_action(env, s, S, offsets[int(np.argmax(Q[context(env, obs)]))])


if __name__ == "__main__":
    hist = load_demand_csv(CSV_PATH)
    n_train = 365
    rlhud.plot(hist["demand"], label="daily demand", title="Full history · train | test split at day 365",
               smooth=7, xlabel="day")
    res = run_pipeline(CSV_PATH, n_train=n_train)
    s, S = res["s_S"]
    rlhud.stat("Tuned on year 1: (s, S)", f"({s}, {S})")
    rows = []
    for k in ("profit", "fill_rate", "lost_sales", "spoiled", "orders"):
        b, a = res["baseline"][k], res["agent"][k]
        rows.append([k, round(b, 3), round(a, 3), f"{100 * (a / b - 1):+.1f} %" if b else "—"])
    rlhud.table(rows, headers=["test year (unseen)", "planner (s, S)", "residual Q", "change"],
                title="Out-of-time evaluation")
    up = res["agent"]["profit"] / res["baseline"]["profit"] - 1
    rlhud.stat("Profit uplift on the test year", 100 * up, unit=" %", good=up > 0)
    rlhud.stat("Agent fill rate", res["agent"]["fill_rate"], good=res["agent"]["fill_rate"] >= 0.9)

    # what does the agent do differently? learned (s, S) shift per context
    Q = res["Q"]
    step = InventoryEnv(demand_series=hist["demand"][:n_train]).order_step
    shift = [float(OFFSETS[np.argmax(Q[c])] * step / 6) for c in range(N_CONTEXTS)]
    rlhud.bars(shift, labels=[f"{DOW_NAMES[c // 2]}{'+P' if c % 2 else ''}" for c in range(N_CONTEXTS)],
               title="Learned shift of (s, S) in units, per context (+P = promo soon)")
    if up <= 0:
        rlhud.text("On this data the agent did not beat the planner out of time. That is a real, "
                   "useful result: check the train/test demand shift and the size of the uplift "
                   "on the training year before trusting any RL policy.", title="Honest result", tone="warn")

## 07 · Beyond the shelf

The same recipe — a simulator calibrated on data, a strong classic baseline, an agent that improves
on it, and an out-of-time evaluation — carries over to many operational problems:

| Problem | State | Action | Reward | Classic baseline |
|---|---|---|---|---|
| Dynamic pricing | stock left, days to season end, demand signals | price level | revenue (− markdown waste) | fixed markdown calendar |
| Ad budget pacing | spend so far, time of day, conversion rates | bid multiplier | conversions − cost | even pacing |
| HVAC / energy | temperatures, occupancy, electricity price | set-points, battery charge | − (energy cost + comfort penalty) | thermostat schedule |
| Fleet rebalancing | vehicles and requests per zone | move vehicles between zones | served trips − driving cost | rebalance to historical demand |

In all of them the hard parts are the ones you met here: a reward that matches the business,
a state that is Markov enough, a baseline that is genuinely tuned, and an evaluation that never
peeks into the future.

## 08 · Debrief

| Step | What you did | Lesson |
|---|---|---|
| 1 | weekday profile, promo lift, tuned (s, S) | understand the data and set the bar |
| 2 | residual Q-learning over (weekday, promo) | learn adjustments, not everything from scratch |
| 3 | train on year 1, evaluate on year 2 | judge on the future, report business metrics |

Ideas to take further:

- Re-run the study with `lead_time=4` or `spoil_rate=0.1` and see which policy degrades more.
- Use `obs="vector"` with linear function approximation or a small DQN, residual or not.
- Run exercise 3 on **ten products** from a public dataset and report the distribution of uplifts.
- Replace ε-greedy exploration with **offline** learning from the planner's own historical
  decisions — the setting most real deployments start from.

---
## 🏁 Debrief
Run the cell below for your mission status, then open the next notebook.

In [ ]:
#@title 🏁 **Mission status** — run to refresh
rlcolab.lesson_footer("15-bonus-inventory")